# INLP Final Project: A100 LLM Router

This notebook trains an offline router for the 11 candidate LLMs. It predicts a configurable cost-aware utility for each `Model_A` ... `Model_K`, ensembles TF-IDF and Qwen3 embedding models, and writes a Kaggle-ready `submission.csv`.

Colab Drive layout: `/content/drive/MyDrive/NLP Final Project/FinalProject_router_a100.ipynb` with data in `/content/drive/MyDrive/NLP Final Project/dataset`.

Default target: exact Kaggle `Reward_0.85 = 0.85 * mean(performance) - 0.15 * mean(cost) / mean(row_max_cost)`. New exact-metric artifacts are written under `Output/router_a100_exact_metric_v2` so the previous `Output/router_a100` public-LB run stays untouched.

In [ ]:
# Colab/A100 setup. Re-run this cell after changing runtimes.
# Keep pandas/numpy compatible with Colab, cuDF, db-dtypes, bqplot, gradio, and numba.
# If this cell downgrades a package after a conflict, restart the runtime before running later cells.
%pip -q install --upgrade-strategy only-if-needed "pandas==2.2.2" "numpy>=1.26.4,<2.1" "transformers>=4.51.0" "sentence-transformers>=2.7.0" accelerate lightgbm scikit-learn scipy tqdm joblib


In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

try:
    import torch
except Exception:
    torch = None

try:
    from lightgbm import LGBMRegressor
except Exception as exc:
    LGBMRegressor = None
    print('LightGBM import failed; TF-IDF Ridge will still run:', repr(exc))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

@dataclass
class CFG:
    seed: int = 42
    project_root: Path = Path.cwd()
    notebook_path: Path = Path('FinalProject_router_a100.ipynb')
    data_dir: Path = Path('dataset')
    out_dir: Path = Path('Output/router_a100_exact_metric_v2')
    previous_run_dir: Path = Path('Output/router_a100')
    previous_public_lb_score: float = 0.45648
    n_splits: int = 5
    smoke_test: bool = False
    smoke_rows: int = 300

    reward_mode: str = 'exact_kaggle_reward_085'
    reward_performance_weight: float = 0.85
    reward_cost_weight: float = 0.15
    cost_denominator: float | None = None

    run_tfidf: bool = True
    run_qwen_embeddings: bool = True
    run_embedding_ridge: bool = True
    run_two_head_qwen_lgbm: bool = True
    run_modernbert_optional: bool = False

    qwen_model_id: str = 'Qwen/Qwen3-Embedding-4B'
    embedding_dim: int = 1024
    embedding_batch_size: int = 8
    chunk_chars: int = 12000
    max_chunks: int = 3
    embedding_cache_name: str = 'qwen3_4b_dim1024_chunks_v1'

    tfidf_max_chars: int = 60000
    tfidf_word_features: int = 120000
    tfidf_char_features: int = 120000
    ridge_alpha: float = 8.0

    lgbm_estimators: int = 900
    lgbm_learning_rate: float = 0.025
    lgbm_num_leaves: int = 31
    lgbm_min_child_samples: int = 30
    embedding_ridge_alpha: float = 4.0

    ensemble_weight_step: float = 0.1
    fallback_model_name: str = 'Model_K'
    fallback_quantiles: tuple[float, ...] = (0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.65, 0.8)

    modernbert_model_id: str = 'answerdotai/ModernBERT-large'
    modernbert_max_length: int = 2048
    modernbert_epochs: int = 2
    modernbert_batch_size: int = 4
    modernbert_grad_accum: int = 4

cfg = CFG()

# Colab Drive layout for this project.
USE_DRIVE = True
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/NLP Final Project')
DRIVE_DATA_DIR = DRIVE_PROJECT_ROOT / 'dataset'
DRIVE_NOTEBOOK_PATH = DRIVE_PROJECT_ROOT / 'FinalProject_router_a100.ipynb'
DRIVE_OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'Output/router_a100_exact_metric_v2'
DRIVE_PREVIOUS_RUN_DIR = DRIVE_PROJECT_ROOT / 'Output/router_a100'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        cfg.project_root = DRIVE_PROJECT_ROOT
        cfg.notebook_path = DRIVE_NOTEBOOK_PATH
        cfg.data_dir = DRIVE_DATA_DIR
        cfg.out_dir = DRIVE_OUTPUT_DIR
        cfg.previous_run_dir = DRIVE_PREVIOUS_RUN_DIR
    except ModuleNotFoundError:
        if DRIVE_DATA_DIR.exists():
            cfg.project_root = DRIVE_PROJECT_ROOT
            cfg.notebook_path = DRIVE_NOTEBOOK_PATH
            cfg.data_dir = DRIVE_DATA_DIR
            cfg.out_dir = DRIVE_OUTPUT_DIR
            cfg.previous_run_dir = DRIVE_PREVIOUS_RUN_DIR
        elif (Path.cwd() / 'dataset' / 'train.csv').exists():
            print('Drive path unavailable outside Colab; using local workspace dataset.')
            cfg.project_root = Path.cwd()
            cfg.notebook_path = Path.cwd() / 'FinalProject_router_a100.ipynb'
            cfg.data_dir = Path.cwd() / 'dataset'
            cfg.out_dir = Path.cwd() / 'Output/router_a100_exact_metric_v2'
            cfg.previous_run_dir = Path.cwd() / 'Output/router_a100'
        else:
            raise FileNotFoundError(f'Expected data at {DRIVE_DATA_DIR} or ./dataset/train.csv')
else:
    if (Path.cwd() / 'dataset' / 'train.csv').exists():
        cfg.project_root = Path.cwd()
        cfg.notebook_path = Path.cwd() / 'FinalProject_router_a100.ipynb'
        cfg.data_dir = Path.cwd() / 'dataset'
        cfg.out_dir = Path.cwd() / 'Output/router_a100_exact_metric_v2'
        cfg.previous_run_dir = Path.cwd() / 'Output/router_a100'
    else:
        cfg.project_root = DRIVE_PROJECT_ROOT
        cfg.notebook_path = DRIVE_NOTEBOOK_PATH
        cfg.data_dir = DRIVE_DATA_DIR
        cfg.out_dir = DRIVE_OUTPUT_DIR
        cfg.previous_run_dir = DRIVE_PREVIOUS_RUN_DIR

cfg.out_dir.mkdir(parents=True, exist_ok=True)
(cfg.out_dir / 'cache').mkdir(parents=True, exist_ok=True)
seed_everything(cfg.seed)

LETTERS = list('ABCDEFGHIJK')
MODEL_NAMES = [f'Model_{letter}' for letter in LETTERS]
PERF_COLS = [f'{name}_performance' for name in MODEL_NAMES]
COST_COLS = [f'{name}_cost' for name in MODEL_NAMES]

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}, indent=2))
if torch is not None:
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
def read_data(cfg: CFG) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_path = cfg.data_dir / 'train.csv'
    test_path = cfg.data_dir / 'test.csv'
    sample_path = cfg.data_dir / 'sample_submission.csv'
    assert train_path.exists(), f'Missing {train_path}'
    assert test_path.exists(), f'Missing {test_path}'
    assert sample_path.exists(), f'Missing {sample_path}'

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sample_df = pd.read_csv(sample_path)

    expected_train = ['ID', 'query'] + [col for pair in zip(PERF_COLS, COST_COLS) for col in pair]
    missing = [col for col in expected_train if col not in train_df.columns]
    assert not missing, f'Missing train columns: {missing}'
    assert list(test_df.columns) == ['ID', 'query'], test_df.columns.tolist()
    assert list(sample_df.columns) == ['ID', 'pred_model'], sample_df.columns.tolist()

    train_df['query'] = train_df['query'].fillna('').astype(str)
    test_df['query'] = test_df['query'].fillna('').astype(str)
    return train_df, test_df, sample_df

train_df, test_df, sample_df = read_data(cfg)
if cfg.smoke_test:
    train_df = train_df.sample(min(cfg.smoke_rows, len(train_df)), random_state=cfg.seed).sort_values('ID').reset_index(drop=True)
    test_df = test_df.head(min(200, len(test_df))).copy()

print('train', train_df.shape, 'test', test_df.shape, 'sample', sample_df.shape)
display(train_df[['ID', 'query']].head())

query_lengths = train_df['query'].str.len()
print('query length summary')
display(query_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame('chars'))

perf = train_df[PERF_COLS].to_numpy(np.float32)
cost = train_df[COST_COLS].to_numpy(np.float32)
summary = pd.DataFrame({
    'model': MODEL_NAMES,
    'mean_performance': perf.mean(axis=0),
    'mean_cost': cost.mean(axis=0),
    'correct_rate': (perf >= 1.0).mean(axis=0),
})
display(summary.sort_values('mean_performance', ascending=False))


In [ ]:
def compute_cost_denominator(cost: np.ndarray) -> float:
    row_max_cost = cost.astype(np.float32).max(axis=1)
    denominator = float(row_max_cost.mean())
    if denominator <= 0:
        raise ValueError('Kaggle cost denominator must be positive')
    return denominator

def compute_reward(performance: np.ndarray, cost: np.ndarray, cfg: CFG) -> np.ndarray:
    """Exact Kaggle Reward_0.85 per candidate, using train mean(row max cost) as the denominator."""
    if cfg.reward_mode != 'exact_kaggle_reward_085':
        raise ValueError(f'Unsupported reward_mode for this notebook: {cfg.reward_mode}')
    if cfg.cost_denominator is None:
        cfg.cost_denominator = compute_cost_denominator(cost)
    performance = performance.astype(np.float32)
    cost = cost.astype(np.float32)
    return cfg.reward_performance_weight * performance - cfg.reward_cost_weight * (cost / cfg.cost_denominator)

def exact_reward_from_means(mean_performance: float, mean_cost: float, cfg: CFG) -> float:
    if cfg.cost_denominator is None:
        raise ValueError('cfg.cost_denominator must be set before evaluation')
    return float(cfg.reward_performance_weight * mean_performance - cfg.reward_cost_weight * (mean_cost / cfg.cost_denominator))

def utility_from_predicted_perf_cost(pred_performance: np.ndarray, pred_cost: np.ndarray, cfg: CFG) -> np.ndarray:
    clipped_cost = np.clip(pred_cost.astype(np.float32), 0.0, None)
    return cfg.reward_performance_weight * pred_performance.astype(np.float32) - cfg.reward_cost_weight * (clipped_cost / cfg.cost_denominator)

def add_numeric_features(df: pd.DataFrame) -> np.ndarray:
    q = df['query'].fillna('').astype(str)
    features = pd.DataFrame({
        'char_len': q.str.len(),
        'word_count': q.str.split().str.len(),
        'line_count': q.str.count('\n') + 1,
        'digit_count': q.str.count(r'\d'),
        'latex_count': q.str.count(r'\$'),
        'code_signal': q.str.count(r'```|def |class |#include|import |SELECT |function |var |const |public |private '),
        'math_signal': q.str.count(r'\\frac|\\sqrt|\\angle|\bequation\b|\bprobability\b|\binteger\b'),
        'question_marks': q.str.count(r'\?'),
    }).astype(np.float32)
    features['log_char_len'] = np.log1p(features['char_len'])
    features['log_word_count'] = np.log1p(features['word_count'])
    return features.to_numpy(np.float32)

def cap_text(text: str, max_chars: int) -> str:
    if len(text) <= max_chars:
        return text
    head = int(max_chars * 0.65)
    tail = max_chars - head
    return text[:head] + '\n\n[... middle truncated ...]\n\n' + text[-tail:]

def build_tfidf_ridge(cfg: CFG) -> Pipeline:
    word = TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        sublinear_tf=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=cfg.tfidf_word_features,
    )
    char = TfidfVectorizer(
        lowercase=True,
        analyzer='char_wb',
        ngram_range=(3, 5),
        min_df=2,
        max_features=cfg.tfidf_char_features,
    )
    return Pipeline([
        ('features', FeatureUnion([('word', word), ('char', char)])),
        ('regressor', MultiOutputRegressor(Ridge(alpha=cfg.ridge_alpha, random_state=cfg.seed))),
    ])

def build_ridge_multioutput(alpha: float, cfg: CFG) -> MultiOutputRegressor:
    return MultiOutputRegressor(Ridge(alpha=alpha, random_state=cfg.seed))

def build_lgbm_multioutput(cfg: CFG) -> MultiOutputRegressor:
    if LGBMRegressor is None:
        raise RuntimeError('LightGBM is not available')
    base = LGBMRegressor(
        objective='regression',
        n_estimators=cfg.lgbm_estimators,
        learning_rate=cfg.lgbm_learning_rate,
        num_leaves=cfg.lgbm_num_leaves,
        min_child_samples=cfg.lgbm_min_child_samples,
        subsample=0.9,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.2,
        random_state=cfg.seed,
        n_jobs=-1,
        verbosity=-1,
    )
    return MultiOutputRegressor(base, n_jobs=1)

cfg.cost_denominator = compute_cost_denominator(cost)
Y = compute_reward(perf, cost, cfg)
oracle_idx = Y.argmax(axis=1)
best_single_idx = int(np.argmax(Y.mean(axis=0)))
print('reward target summary')
print('train mean row max cost denominator', cfg.cost_denominator)
print('oracle mean reward', float(Y[np.arange(len(Y)), oracle_idx].mean()))
print('best single model', MODEL_NAMES[best_single_idx], float(Y[:, best_single_idx].mean()))


In [ ]:
def evaluate_route_indices(name: str, pred_idx: np.ndarray, performance: np.ndarray, cost: np.ndarray, reward: np.ndarray, cfg: CFG) -> dict:
    pred_idx = np.asarray(pred_idx, dtype=np.int64)
    rows = np.arange(len(pred_idx))
    counts = np.bincount(pred_idx, minlength=len(MODEL_NAMES))
    mean_performance = float(performance[rows, pred_idx].mean())
    mean_cost = float(cost[rows, pred_idx].mean())
    exact_reward = exact_reward_from_means(mean_performance, mean_cost, cfg)
    result = {
        'name': name,
        'mean_reward': exact_reward,
        'exact_reward': exact_reward,
        'reward_from_matrix': float(reward[rows, pred_idx].mean()),
        'mean_performance': mean_performance,
        'mean_cost': mean_cost,
        'model_counts': json.dumps({MODEL_NAMES[i]: int(counts[i]) for i in range(len(MODEL_NAMES))}),
    }
    return result

def save_prediction_artifacts(name: str, oof_scores: np.ndarray, test_scores: np.ndarray, cfg: CFG) -> None:
    np.savez_compressed(cfg.out_dir / f'{name}_oof_predictions.npz', scores=oof_scores.astype(np.float32), model_names=np.asarray(MODEL_NAMES))
    np.savez_compressed(cfg.out_dir / f'{name}_test_predictions.npz', scores=test_scores.astype(np.float32), model_names=np.asarray(MODEL_NAMES))

def cross_validate_regressor(name: str, estimator, X, Y: np.ndarray, cfg: CFG) -> tuple[np.ndarray, dict]:
    kf = KFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
    oof = np.zeros_like(Y, dtype=np.float32)
    fold_rows = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(Y))), 1):
        model = clone(estimator)
        X_tr = X[tr_idx] if not isinstance(X, list) else [X[i] for i in tr_idx]
        X_va = X[va_idx] if not isinstance(X, list) else [X[i] for i in va_idx]
        model.fit(X_tr, Y[tr_idx])
        pred = model.predict(X_va).astype(np.float32)
        oof[va_idx] = pred
        route_idx = pred.argmax(axis=1)
        fold_metrics = evaluate_route_indices(name, route_idx, perf[va_idx], cost[va_idx], Y[va_idx], cfg)
        fold_reward = fold_metrics['mean_reward']
        fold_rmse = math.sqrt(mean_squared_error(Y[va_idx].ravel(), pred.ravel()))
        fold_rows.append({'model': name, 'fold': fold, 'route_reward': float(fold_reward), 'mean_performance': fold_metrics['mean_performance'], 'mean_cost': fold_metrics['mean_cost'], 'utility_rmse': float(fold_rmse)})
        print(f'{name} fold {fold}: route_reward={fold_reward:.6f} utility_rmse={fold_rmse:.6f}')
        del model
        gc.collect()
    cv_df = pd.DataFrame(fold_rows)
    cv_df.to_csv(cfg.out_dir / f'{name}_cv_folds.csv', index=False)
    metrics = evaluate_route_indices(name, oof.argmax(axis=1), perf, cost, Y, cfg)
    metrics['utility_rmse'] = float(math.sqrt(mean_squared_error(Y.ravel(), oof.ravel())))
    return oof, metrics

def cross_validate_two_head(name: str, perf_estimator, cost_estimator, X, performance: np.ndarray, cost_targets: np.ndarray, cfg: CFG) -> tuple[np.ndarray, dict]:
    kf = KFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
    oof_scores = np.zeros_like(Y, dtype=np.float32)
    oof_perf = np.zeros_like(performance, dtype=np.float32)
    oof_cost = np.zeros_like(cost_targets, dtype=np.float32)
    fold_rows = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(Y))), 1):
        perf_model = clone(perf_estimator)
        cost_model = clone(cost_estimator)
        X_tr = X[tr_idx] if not isinstance(X, list) else [X[i] for i in tr_idx]
        X_va = X[va_idx] if not isinstance(X, list) else [X[i] for i in va_idx]
        perf_model.fit(X_tr, performance[tr_idx])
        cost_model.fit(X_tr, cost_targets[tr_idx])
        pred_perf = perf_model.predict(X_va).astype(np.float32)
        pred_cost = cost_model.predict(X_va).astype(np.float32)
        pred_scores = utility_from_predicted_perf_cost(pred_perf, pred_cost, cfg)
        oof_perf[va_idx] = pred_perf
        oof_cost[va_idx] = pred_cost
        oof_scores[va_idx] = pred_scores
        route_idx = pred_scores.argmax(axis=1)
        fold_metrics = evaluate_route_indices(name, route_idx, perf[va_idx], cost[va_idx], Y[va_idx], cfg)
        utility_rmse = math.sqrt(mean_squared_error(Y[va_idx].ravel(), pred_scores.ravel()))
        perf_rmse = math.sqrt(mean_squared_error(performance[va_idx].ravel(), pred_perf.ravel()))
        cost_rmse = math.sqrt(mean_squared_error(cost_targets[va_idx].ravel(), pred_cost.ravel()))
        fold_rows.append({'model': name, 'fold': fold, 'route_reward': fold_metrics['mean_reward'], 'mean_performance': fold_metrics['mean_performance'], 'mean_cost': fold_metrics['mean_cost'], 'utility_rmse': utility_rmse, 'performance_rmse': perf_rmse, 'cost_rmse': cost_rmse})
        print(f'{name} fold {fold}: route_reward={fold_metrics["mean_reward"]:.6f} utility_rmse={utility_rmse:.6f}')
        del perf_model, cost_model
        gc.collect()
    pd.DataFrame(fold_rows).to_csv(cfg.out_dir / f'{name}_cv_folds.csv', index=False)
    metrics = evaluate_route_indices(name, oof_scores.argmax(axis=1), perf, cost, Y, cfg)
    metrics['utility_rmse'] = float(math.sqrt(mean_squared_error(Y.ravel(), oof_scores.ravel())))
    metrics['performance_rmse'] = float(math.sqrt(mean_squared_error(performance.ravel(), oof_perf.ravel())))
    metrics['cost_rmse'] = float(math.sqrt(mean_squared_error(cost_targets.ravel(), oof_cost.ravel())))
    return oof_scores, metrics

def rank_blend(prediction_mats: list[np.ndarray], weights: list[float] | None = None) -> np.ndarray:
    if weights is None:
        weights = [1.0] * len(prediction_mats)
    weights = np.asarray(weights, dtype=np.float32)
    weights = weights / weights.sum()
    blended = np.zeros_like(prediction_mats[0], dtype=np.float32)
    for mat, weight in zip(prediction_mats, weights):
        ranks = np.vstack([rankdata(row, method='average') for row in mat]).astype(np.float32)
        blended += weight * ranks
    return blended

def weighted_score_blend(prediction_mats: list[np.ndarray], weights: list[float]) -> np.ndarray:
    weights_np = np.asarray(weights, dtype=np.float32)
    weights_np = weights_np / weights_np.sum()
    blended = np.zeros_like(prediction_mats[0], dtype=np.float32)
    for mat, weight in zip(prediction_mats, weights_np):
        blended += weight * mat.astype(np.float32)
    return blended

def iter_weight_grid(n_models: int, step: float):
    units = int(round(1.0 / step))
    weights = [0] * n_models
    def rec(pos: int, remaining: int):
        if pos == n_models - 1:
            weights[pos] = remaining
            yield [w / units for w in weights]
            return
        for value in range(remaining + 1):
            weights[pos] = value
            yield from rec(pos + 1, remaining - value)
    yield from rec(0, units)

def apply_margin_fallback(scores: np.ndarray, fallback_idx: int, threshold: float) -> np.ndarray:
    if scores.shape[1] < 2:
        return np.full(scores.shape[0], fallback_idx, dtype=np.int64)
    top2 = np.sort(np.partition(scores, -2, axis=1)[:, -2:], axis=1)
    margins = top2[:, 1] - top2[:, 0]
    pred_idx = scores.argmax(axis=1).astype(np.int64)
    pred_idx[margins <= threshold] = fallback_idx
    return pred_idx

def validate_expected_reward_baselines(cfg: CFG) -> None:
    model_k_idx = MODEL_NAMES.index('Model_K')
    model_k_reward = evaluate_route_indices('always_Model_K_check', np.full(len(train_df), model_k_idx), perf, cost, Y, cfg)['mean_reward']
    oracle_reward = evaluate_route_indices('oracle_check', Y.argmax(axis=1), perf, cost, Y, cfg)['mean_reward']
    print(f'exact metric check: always Model_K={model_k_reward:.6f} expected about 0.450376')
    print(f'exact metric check: oracle={oracle_reward:.6f} expected about 0.673071')
    assert abs(model_k_reward - 0.450376) < 0.002, model_k_reward
    assert abs(oracle_reward - 0.673071) < 0.002, oracle_reward

validate_expected_reward_baselines(cfg)
baseline_rows = []
for model_idx, model_name in enumerate(MODEL_NAMES):
    baseline_rows.append(evaluate_route_indices(f'always_{model_name}', np.full(len(train_df), model_idx), perf, cost, Y, cfg))
baseline_rows.append(evaluate_route_indices('oracle', Y.argmax(axis=1), perf, cost, Y, cfg))
display(pd.DataFrame(baseline_rows).sort_values('mean_reward', ascending=False).head(15))


In [ ]:
oof_predictions: dict[str, np.ndarray] = {}
test_predictions: dict[str, np.ndarray] = {}
model_metrics: list[dict] = []

if cfg.run_tfidf:
    print('Training TF-IDF + Ridge baseline')
    X_text = [cap_text(text, cfg.tfidf_max_chars) for text in train_df['query'].tolist()]
    X_test_text = [cap_text(text, cfg.tfidf_max_chars) for text in test_df['query'].tolist()]
    tfidf_estimator = build_tfidf_ridge(cfg)
    tfidf_oof, tfidf_metrics = cross_validate_regressor('tfidf_ridge', tfidf_estimator, X_text, Y, cfg)
    oof_predictions['tfidf_ridge'] = tfidf_oof
    model_metrics.append(tfidf_metrics)

    final_tfidf = build_tfidf_ridge(cfg)
    final_tfidf.fit(X_text, Y)
    test_predictions['tfidf_ridge'] = final_tfidf.predict(X_test_text).astype(np.float32)
    save_prediction_artifacts('tfidf_ridge', tfidf_oof, test_predictions['tfidf_ridge'], cfg)
    joblib.dump(final_tfidf, cfg.out_dir / 'tfidf_ridge.joblib')
    del final_tfidf
    gc.collect()

display(pd.DataFrame(model_metrics).sort_values('mean_reward', ascending=False) if model_metrics else pd.DataFrame())


In [ ]:
def make_chunks(text: str, cfg: CFG) -> list[str]:
    text = text or ''
    if len(text) <= cfg.chunk_chars or cfg.max_chunks <= 1:
        return [text]
    chunks = [text[:cfg.chunk_chars], text[-cfg.chunk_chars:]]
    if cfg.max_chunks >= 3:
        midpoint = max(0, len(text) // 2 - cfg.chunk_chars // 2)
        chunks.insert(1, text[midpoint:midpoint + cfg.chunk_chars])
    seen = set()
    unique_chunks = []
    for chunk in chunks[:cfg.max_chunks]:
        if chunk not in seen:
            unique_chunks.append(chunk)
            seen.add(chunk)
    return unique_chunks

def load_sentence_transformer(cfg: CFG):
    from sentence_transformers import SentenceTransformer
    common_kwargs = {
        'model_name_or_path': cfg.qwen_model_id,
        'truncate_dim': cfg.embedding_dim,
        'trust_remote_code': True,
    }
    if torch is not None and torch.cuda.is_available():
        try:
            return SentenceTransformer(
                **common_kwargs,
                model_kwargs={'torch_dtype': torch.float16, 'attn_implementation': 'flash_attention_2', 'device_map': 'auto'},
                tokenizer_kwargs={'padding_side': 'left'},
            )
        except Exception as exc:
            print('flash_attention_2 load failed; falling back to sdpa/default attention:', repr(exc))
            return SentenceTransformer(
                **common_kwargs,
                model_kwargs={'torch_dtype': torch.float16, 'device_map': 'auto'},
                tokenizer_kwargs={'padding_side': 'left'},
            )
    return SentenceTransformer(**common_kwargs)

def embed_queries(df: pd.DataFrame, split_name: str, cfg: CFG) -> np.ndarray:
    cache_name = f'{cfg.embedding_cache_name}_{split_name}_{len(df)}.npy'
    cache_path = cfg.out_dir / 'cache' / cache_name
    if cache_path.exists():
        print('Loading cached embeddings:', cache_path)
        return np.load(cache_path)
    previous_cache_candidates = [
        cfg.previous_run_dir / 'cache' / cache_name,
        cfg.project_root / 'Output/router_a100/cache' / cache_name,
        cfg.project_root / 'outputs/router_a100/cache' / cache_name,
    ]
    seen_cache_paths = set()
    for previous_cache_path in previous_cache_candidates:
        if previous_cache_path in seen_cache_paths:
            continue
        seen_cache_paths.add(previous_cache_path)
        if previous_cache_path.exists():
            print('Reusing previous run embeddings:', previous_cache_path)
            emb = np.load(previous_cache_path)
            np.save(cache_path, emb)
            print('Copied embeddings into exact-metric cache:', cache_path)
            return emb

    model = load_sentence_transformer(cfg)
    flat_chunks: list[str] = []
    owners: list[int] = []
    for row_idx, text in enumerate(df['query'].fillna('').astype(str).tolist()):
        chunks = make_chunks(text, cfg)
        flat_chunks.extend(chunks)
        owners.extend([row_idx] * len(chunks))
    print(f'Embedding {split_name}: {len(df)} queries -> {len(flat_chunks)} chunks')
    chunk_emb = model.encode(
        flat_chunks,
        batch_size=cfg.embedding_batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    owners_np = np.asarray(owners, dtype=np.int64)
    emb = np.zeros((len(df), chunk_emb.shape[1]), dtype=np.float32)
    counts = np.zeros(len(df), dtype=np.float32)
    np.add.at(emb, owners_np, chunk_emb)
    np.add.at(counts, owners_np, 1.0)
    emb /= np.maximum(counts[:, None], 1.0)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    emb = emb / np.maximum(norms, 1e-12)
    np.save(cache_path, emb)
    del model, chunk_emb
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Saved embeddings:', cache_path, emb.shape)
    return emb

if cfg.run_qwen_embeddings:
    train_emb = embed_queries(train_df, 'train', cfg)
    test_emb = embed_queries(test_df, 'test', cfg)
    scaler = StandardScaler()
    X_num_train = scaler.fit_transform(add_numeric_features(train_df)).astype(np.float32)
    X_num_test = scaler.transform(add_numeric_features(test_df)).astype(np.float32)
    X_emb_train = np.hstack([train_emb, X_num_train]).astype(np.float32)
    X_emb_test = np.hstack([test_emb, X_num_test]).astype(np.float32)

    qwen_estimator = build_lgbm_multioutput(cfg)
    qwen_oof, qwen_metrics = cross_validate_regressor('qwen3_embedding_lgbm', qwen_estimator, X_emb_train, Y, cfg)
    oof_predictions['qwen3_embedding_lgbm'] = qwen_oof
    model_metrics.append(qwen_metrics)

    final_qwen = build_lgbm_multioutput(cfg)
    final_qwen.fit(X_emb_train, Y)
    test_predictions['qwen3_embedding_lgbm'] = final_qwen.predict(X_emb_test).astype(np.float32)
    save_prediction_artifacts('qwen3_embedding_lgbm', qwen_oof, test_predictions['qwen3_embedding_lgbm'], cfg)
    joblib.dump({'model': final_qwen, 'scaler': scaler, 'cfg': asdict(cfg)}, cfg.out_dir / 'qwen3_embedding_lgbm.joblib')
    del final_qwen
    gc.collect()

    if cfg.run_embedding_ridge:
        print('Training Qwen embeddings + Ridge utility regressor')
        embedding_ridge = build_ridge_multioutput(cfg.embedding_ridge_alpha, cfg)
        ridge_oof, ridge_metrics = cross_validate_regressor('qwen3_embedding_ridge', embedding_ridge, X_emb_train, Y, cfg)
        oof_predictions['qwen3_embedding_ridge'] = ridge_oof
        model_metrics.append(ridge_metrics)
        final_ridge = build_ridge_multioutput(cfg.embedding_ridge_alpha, cfg)
        final_ridge.fit(X_emb_train, Y)
        test_predictions['qwen3_embedding_ridge'] = final_ridge.predict(X_emb_test).astype(np.float32)
        save_prediction_artifacts('qwen3_embedding_ridge', ridge_oof, test_predictions['qwen3_embedding_ridge'], cfg)
        joblib.dump({'model': final_ridge, 'scaler': scaler, 'cfg': asdict(cfg)}, cfg.out_dir / 'qwen3_embedding_ridge.joblib')
        del final_ridge
        gc.collect()

    if cfg.run_two_head_qwen_lgbm:
        print('Training Qwen embeddings + two-head LightGBM performance/cost router')
        perf_estimator = build_lgbm_multioutput(cfg)
        cost_estimator = build_lgbm_multioutput(cfg)
        two_head_oof, two_head_metrics = cross_validate_two_head('qwen3_embedding_lgbm_two_head', perf_estimator, cost_estimator, X_emb_train, perf, cost, cfg)
        oof_predictions['qwen3_embedding_lgbm_two_head'] = two_head_oof
        model_metrics.append(two_head_metrics)
        final_perf = build_lgbm_multioutput(cfg)
        final_cost = build_lgbm_multioutput(cfg)
        final_perf.fit(X_emb_train, perf)
        final_cost.fit(X_emb_train, cost)
        pred_perf_test = final_perf.predict(X_emb_test).astype(np.float32)
        pred_cost_test = final_cost.predict(X_emb_test).astype(np.float32)
        test_predictions['qwen3_embedding_lgbm_two_head'] = utility_from_predicted_perf_cost(pred_perf_test, pred_cost_test, cfg)
        save_prediction_artifacts('qwen3_embedding_lgbm_two_head', two_head_oof, test_predictions['qwen3_embedding_lgbm_two_head'], cfg)
        joblib.dump({'performance_model': final_perf, 'cost_model': final_cost, 'scaler': scaler, 'cfg': asdict(cfg)}, cfg.out_dir / 'qwen3_embedding_lgbm_two_head.joblib')
        del final_perf, final_cost, pred_perf_test, pred_cost_test
        gc.collect()

display(pd.DataFrame(model_metrics).sort_values('mean_reward', ascending=False) if model_metrics else pd.DataFrame())


In [ ]:
# Optional A100 experiment: ModernBERT exact-utility regression with OOF/test predictions.
# This is disabled by default because it trains one model per fold plus a final full-data model.
if cfg.run_modernbert_optional:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', 'datasets', 'evaluate'])
    import torch
    from datasets import Dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

    tokenizer = AutoTokenizer.from_pretrained(cfg.modernbert_model_id)

    def make_modernbert_dataset(queries: pd.Series, labels: np.ndarray | None = None) -> Dataset:
        data = {'query': queries.fillna('').astype(str).tolist()}
        if labels is not None:
            data['labels'] = labels.astype(np.float32).tolist()
        ds = Dataset.from_dict(data)
        def tokenize(batch):
            return tokenizer(batch['query'], truncation=True, max_length=cfg.modernbert_max_length)
        return ds.map(tokenize, batched=True, remove_columns=['query'])

    def load_modernbert_model():
        kwargs = {'num_labels': len(MODEL_NAMES), 'problem_type': 'regression'}
        if torch is not None and torch.cuda.is_available():
            kwargs['torch_dtype'] = torch.bfloat16
        return AutoModelForSequenceClassification.from_pretrained(cfg.modernbert_model_id, **kwargs)

    modernbert_oof = np.zeros_like(Y, dtype=np.float32)
    fold_rows = []
    kf = KFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(train_df))), 1):
        train_ds = make_modernbert_dataset(train_df.iloc[tr_idx]['query'], Y[tr_idx])
        va_ds = make_modernbert_dataset(train_df.iloc[va_idx]['query'], Y[va_idx])
        model = load_modernbert_model()
        args = TrainingArguments(
            output_dir=str(cfg.out_dir / f'modernbert_large/fold_{fold}'),
            learning_rate=1e-5,
            num_train_epochs=cfg.modernbert_epochs,
            per_device_train_batch_size=cfg.modernbert_batch_size,
            gradient_accumulation_steps=cfg.modernbert_grad_accum,
            bf16=bool(torch is not None and torch.cuda.is_available()),
            save_strategy='no',
            logging_steps=25,
            report_to='none',
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=DataCollatorWithPadding(tokenizer))
        trainer.train()
        pred = trainer.predict(va_ds).predictions.astype(np.float32)
        modernbert_oof[va_idx] = pred
        route_idx = pred.argmax(axis=1)
        fold_metrics = evaluate_route_indices('modernbert_large', route_idx, perf[va_idx], cost[va_idx], Y[va_idx], cfg)
        fold_rmse = math.sqrt(mean_squared_error(Y[va_idx].ravel(), pred.ravel()))
        fold_rows.append({'model': 'modernbert_large', 'fold': fold, 'route_reward': fold_metrics['mean_reward'], 'mean_performance': fold_metrics['mean_performance'], 'mean_cost': fold_metrics['mean_cost'], 'utility_rmse': fold_rmse})
        print(f'modernbert_large fold {fold}: route_reward={fold_metrics["mean_reward"]:.6f} utility_rmse={fold_rmse:.6f}')
        del trainer, model, train_ds, va_ds, pred
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pd.DataFrame(fold_rows).to_csv(cfg.out_dir / 'modernbert_large_cv_folds.csv', index=False)
    mb_metrics = evaluate_route_indices('modernbert_large', modernbert_oof.argmax(axis=1), perf, cost, Y, cfg)
    mb_metrics['utility_rmse'] = float(math.sqrt(mean_squared_error(Y.ravel(), modernbert_oof.ravel())))
    oof_predictions['modernbert_large'] = modernbert_oof
    model_metrics.append(mb_metrics)

    full_ds = make_modernbert_dataset(train_df['query'], Y)
    test_ds = make_modernbert_dataset(test_df['query'])
    final_model = load_modernbert_model()
    args = TrainingArguments(
        output_dir=str(cfg.out_dir / 'modernbert_large/final_train'),
        learning_rate=1e-5,
        num_train_epochs=cfg.modernbert_epochs,
        per_device_train_batch_size=cfg.modernbert_batch_size,
        gradient_accumulation_steps=cfg.modernbert_grad_accum,
        bf16=bool(torch is not None and torch.cuda.is_available()),
        save_strategy='epoch',
        logging_steps=25,
        report_to='none',
    )
    trainer = Trainer(model=final_model, args=args, train_dataset=full_ds, data_collator=DataCollatorWithPadding(tokenizer))
    trainer.train()
    test_predictions['modernbert_large'] = trainer.predict(test_ds).predictions.astype(np.float32)
    trainer.save_model(str(cfg.out_dir / 'modernbert_large/final'))
    tokenizer.save_pretrained(str(cfg.out_dir / 'modernbert_large/final'))
    save_prediction_artifacts('modernbert_large', modernbert_oof, test_predictions['modernbert_large'], cfg)
    display(pd.DataFrame(model_metrics).sort_values('mean_reward', ascending=False))


In [ ]:
assert test_predictions, 'No test predictions were created. Enable cfg.run_tfidf, cfg.run_qwen_embeddings, or cfg.run_modernbert_optional.'
assert set(oof_predictions) == set(test_predictions), f'OOF/test prediction mismatch: {set(oof_predictions)} vs {set(test_predictions)}'

def metric_for_scores(name: str, scores: np.ndarray, source: str, extra: dict | None = None) -> dict:
    metrics = evaluate_route_indices(name, scores.argmax(axis=1), perf, cost, Y, cfg)
    metrics['source'] = source
    if extra:
        metrics.update(extra)
    return metrics

def make_submission(file_name: str, pred_idx: np.ndarray) -> Path:
    pred_model = [MODEL_NAMES[int(i)] for i in pred_idx]
    submission = pd.DataFrame({'ID': test_df['ID'].to_numpy(), 'pred_model': pred_model})
    assert list(submission.columns) == ['ID', 'pred_model']
    assert len(submission) == len(test_df) == len(sample_df)
    assert submission['pred_model'].isin(MODEL_NAMES).all()
    assert submission['ID'].tolist() == sample_df['ID'].tolist()
    path = cfg.out_dir / file_name
    submission.to_csv(path, index=False)
    distribution = submission['pred_model'].value_counts().reindex(MODEL_NAMES, fill_value=0)
    distribution_path = path.with_name(f'{path.stem}_model_distribution.txt')
    with open(distribution_path, 'w') as f:
        f.write(f'Model distribution for {path.name}\n')
        f.write(f'Total rows: {len(submission)}\n\n')
        f.write('model,count,percent\n')
        for model_name, count in distribution.items():
            f.write(f'{model_name},{int(count)},{count / len(submission) * 100:.4f}%\n')
    print('Wrote', path)
    print('Wrote', distribution_path)
    display(distribution.rename_axis('pred_model').reset_index(name='count'))
    return path

comparison_rows = baseline_rows + model_metrics.copy()
candidate_records: dict[str, dict] = {}

for name, scores in oof_predictions.items():
    metrics = metric_for_scores(name, scores, 'single')
    candidate_records[name] = {'name': name, 'oof_scores': scores, 'test_scores': test_predictions[name], 'metrics': metrics, 'kind': 'single'}

names = list(oof_predictions.keys())
ensemble_records: dict[str, dict] = {}
if len(names) >= 2:
    mats = [oof_predictions[name] for name in names]
    test_mats = [test_predictions[name] for name in names]
    best_weighted = None
    for weights in iter_weight_grid(len(names), cfg.ensemble_weight_step):
        if sum(weight > 0 for weight in weights) < 2:
            continue
        scores = weighted_score_blend(mats, weights)
        weight_map = {name: round(float(weight), 4) for name, weight in zip(names, weights) if weight > 0}
        metrics = metric_for_scores('weighted_ensemble_grid', scores, 'ensemble', {'weights': json.dumps(weight_map)})
        if best_weighted is None or metrics['mean_reward'] > best_weighted['metrics']['mean_reward']:
            best_weighted = {'weights': weights, 'weight_map': weight_map, 'oof_scores': scores, 'metrics': metrics}
    if best_weighted is not None:
        weighted_test = weighted_score_blend(test_mats, best_weighted['weights'])
        weighted_metrics = best_weighted['metrics']
        weighted_metrics['name'] = 'weighted_ensemble_best'
        ensemble_records['weighted_ensemble_best'] = {'name': 'weighted_ensemble_best', 'oof_scores': best_weighted['oof_scores'], 'test_scores': weighted_test, 'metrics': weighted_metrics, 'kind': 'ensemble'}
        comparison_rows.append(weighted_metrics)
        save_prediction_artifacts('weighted_ensemble_best', best_weighted['oof_scores'], weighted_test, cfg)

    rank_oof = rank_blend(mats)
    rank_test = rank_blend(test_mats)
    rank_metrics = metric_for_scores('rank_blend_oof', rank_oof, 'ensemble')
    ensemble_records['rank_blend_oof'] = {'name': 'rank_blend_oof', 'oof_scores': rank_oof, 'test_scores': rank_test, 'metrics': rank_metrics, 'kind': 'ensemble'}
    comparison_rows.append(rank_metrics)
    save_prediction_artifacts('rank_blend_oof', rank_oof, rank_test, cfg)

candidate_records.update(ensemble_records)
best_single = max((record for record in candidate_records.values() if record['kind'] == 'single'), key=lambda record: record['metrics']['mean_reward'])
best_ensemble = max((record for record in candidate_records.values() if record['kind'] == 'ensemble'), key=lambda record: record['metrics']['mean_reward'], default=best_single)

fallback_idx = MODEL_NAMES.index(cfg.fallback_model_name)
base_for_fallback = best_ensemble
top2 = np.sort(np.partition(base_for_fallback['oof_scores'], -2, axis=1)[:, -2:], axis=1)
margins = top2[:, 1] - top2[:, 0]
thresholds = sorted(set(float(np.quantile(margins, q)) for q in cfg.fallback_quantiles))
best_fallback = None
for threshold in thresholds:
    pred_idx = apply_margin_fallback(base_for_fallback['oof_scores'], fallback_idx, threshold)
    metrics = evaluate_route_indices(f'{base_for_fallback["name"]}_fallback_{cfg.fallback_model_name}', pred_idx, perf, cost, Y, cfg)
    metrics['source'] = 'fallback'
    metrics['base_candidate'] = base_for_fallback['name']
    metrics['fallback_model'] = cfg.fallback_model_name
    metrics['fallback_threshold'] = threshold
    if best_fallback is None or metrics['mean_reward'] > best_fallback['metrics']['mean_reward']:
        best_fallback = {'name': metrics['name'], 'threshold': threshold, 'metrics': metrics}

fallback_test_idx = apply_margin_fallback(base_for_fallback['test_scores'], fallback_idx, best_fallback['threshold'])
fallback_record = {'name': 'ensemble_model_k_fallback', 'oof_scores': base_for_fallback['oof_scores'], 'test_scores': base_for_fallback['test_scores'], 'test_pred_idx': fallback_test_idx, 'metrics': best_fallback['metrics'], 'kind': 'fallback'}
candidate_records[fallback_record['name']] = fallback_record
comparison_rows.append(best_fallback['metrics'])

comparison_df = pd.DataFrame(comparison_rows).sort_values('mean_reward', ascending=False)
comparison_df.to_csv(cfg.out_dir / 'model_comparison_exact.csv', index=False)
comparison_df.to_csv(cfg.out_dir / 'model_comparison.csv', index=False)
display(comparison_df)

single_test_idx = best_single['test_scores'].argmax(axis=1)
ensemble_test_idx = best_ensemble['test_scores'].argmax(axis=1)
make_submission('submission_exact_single_best.csv', single_test_idx)
make_submission('submission_exact_ensemble_best.csv', ensemble_test_idx)
make_submission('submission_exact_ensemble_model_k_fallback.csv', fallback_test_idx)

submission_choices = [best_single, best_ensemble, fallback_record]
best_overall = max(submission_choices, key=lambda record: record['metrics']['mean_reward'])
if best_overall['kind'] == 'fallback':
    best_test_idx = best_overall['test_pred_idx']
else:
    best_test_idx = best_overall['test_scores'].argmax(axis=1)
make_submission('submission_exact_best.csv', best_test_idx)
make_submission('submission.csv', best_test_idx)
print('Best exact-CV submission candidate:', best_overall['name'], best_overall['metrics'])

with open(cfg.out_dir / 'run_config.json', 'w') as f:
    json.dump({k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}, f, indent=2)

print('Artifacts in', cfg.out_dir)
print('\n'.join(str(p.relative_to(cfg.out_dir)) for p in sorted(cfg.out_dir.rglob('*')) if p.is_file()))
